#### 파일을 읽을때는 용량이 크기때문에 chunk로 진행
- 한 번에 읽지 말고 여러번에 나눠서 읽기 처리

1. category_code 결측패턴확인
2. brand 결측확인
3. 가격 이상치 분포 확인

#### 라이브러리

In [13]:
import pandas as pd
from collections import Counter, defaultdict
import numpy as np

#### 파일 불러오기

In [14]:
# 분석할 파일
files = [
    "2019-Oct.csv",
    "2019-Nov.csv"
]

chunksize = 500000

#### 10월/11월 정보 확인

In [15]:
# 파일 읽기(chunk로 읽기)
oct_reader = pd.read_csv(files[0], chunksize=chunksize)
nov_reader = pd.read_csv(files[1], chunksize=chunksize)

# 그 중 첫 번째 chunk만 가져오기 
oct_chunk = next(oct_reader)
nov_chunk = next(nov_reader)

# 10월
display(oct_chunk.head())
print(oct_chunk.dtypes)
print("\n")
oct_chunk.info()

# 11월
display(nov_chunk.head())
print(nov_chunk.dtypes)
print("\n")
nov_chunk.info()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


event_time           str
event_type           str
product_id         int64
category_id        int64
category_code        str
brand                str
price            float64
user_id            int64
user_session         str
dtype: object


<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     500000 non-null  str    
 1   event_type     500000 non-null  str    
 2   product_id     500000 non-null  int64  
 3   category_id    500000 non-null  int64  
 4   category_code  341561 non-null  str    
 5   brand          428149 non-null  str    
 6   price          500000 non-null  float64
 7   user_id        500000 non-null  int64  
 8   user_session   500000 non-null  str    
dtypes: float64(1), int64(3), str(5)
memory usage: 34.3 MB


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


event_time           str
event_type           str
product_id         int64
category_id        int64
category_code        str
brand                str
price            float64
user_id            int64
user_session         str
dtype: object


<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     500000 non-null  str    
 1   event_type     500000 non-null  str    
 2   product_id     500000 non-null  int64  
 3   category_id    500000 non-null  int64  
 4   category_code  337628 non-null  str    
 5   brand          425209 non-null  str    
 6   price          500000 non-null  float64
 7   user_id        500000 non-null  int64  
 8   user_session   500000 non-null  str    
dtypes: float64(1), int64(3), str(5)
memory usage: 34.3 MB


In [16]:
# 전체 행 수 
oct_rows = len(oct_chunk)   # 첫 chunk 행 수부터 시작
nov_rows = len(nov_chunk) 

for chunk in oct_reader:
    oct_rows += len(chunk)

for chunk in nov_reader:
    nov_rows += len(chunk)

print("10월 전체 행 수:", oct_rows)
print("11월 전체 행 수:", nov_rows)

10월 전체 행 수: 42448764
11월 전체 행 수: 67501979


#### 월별 전체 / 스마트폰 정리

In [17]:
summary = []

for file in files:
    total_rows = 0
    category_missing = 0
    brand_missing = 0
    smartphone_rows = 0
    nonpositive_price = 0

    for chunk in pd.read_csv(file, chunksize=chunksize):
        total_rows += len(chunk)

        category_missing += chunk["category_code"].isna().sum()
        brand_missing += chunk["brand"].isna().sum()
        smartphone_rows += chunk["category_code"].astype("string").str.contains("smartphone", case=False, na=False).sum()
        nonpositive_price += (pd.to_numeric(chunk["price"], errors="coerce") <= 0).sum()

    summary.append({
        "file": file,
        "rows": total_rows,
        "category_code 결측 수": category_missing,
        "category_code 결측 비율": round(category_missing / total_rows * 100, 2),
        "brand 결측 수": brand_missing,
        "brand 결측 비율": round(brand_missing / total_rows * 100, 2),
        "smartphone 행 수": smartphone_rows,
        "smartphone 비율": round(smartphone_rows / total_rows * 100, 2),
        "price <=0": nonpositive_price
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

,file,rows,category_code 결측 수,category_code 결측 비율,brand 결측 수,brand 결측 비율,smartphone 행 수,smartphone 비율,price <=0
0,2019-Oct.csv,42448764,13515609,31.84,6117080,14.41,11507231,27.11,68673
1,2019-Nov.csv,67501979,21898171,32.44,9224078,13.66,16375000,24.26,188088


- category_code 결측 비율이 높음
  - 31~32%로 상당히 높음
  - category_code를 분석해봐야함(패턴 확인)
  - product_id로 구별해 카테고리를 설정해보는건 어떤지
- brand 결측 비율도 적지 않음
  - 13~14%로 지나치기에는 애매
  - 하지만 brand는 'unknown' 처리로 괜찮아 보임
- smartphone 비율 (전체 기준)
  - 24~27%로 50% 이상은 아님
  - 물론 비율이 크긴 함
  - 그래서 전체 분석에서 스마트폰/그외 이걸로 나누기에는 조금 애매 할 수 있음
- price이 0이하 값이 있음
  - 전자상거래 데이터에서 가격이 0인것은 데이터 누락이 아닐까?
  - 누락이라면 제거 할 필요가 있음

category_code 결측 패턴 확인

In [18]:
result = []

for file in files:
    event_total = {}
    event_missing = {}

    for chunk in pd.read_csv(file, chunksize=chunksize):
        # event_type별 전체 건수
        total_counts = chunk["event_type"].value_counts(dropna=False).to_dict()
        for k, v in total_counts.items():
            event_total[k] = event_total.get(k, 0) + v

        # category_code 결측인 행만 따로
        missing_chunk = chunk[chunk["category_code"].isna()]
        missing_counts = missing_chunk["event_type"].value_counts(dropna=False).to_dict()
        for k, v in missing_counts.items():
            event_missing[k] = event_missing.get(k, 0) + v

    # event_type별 정리
    all_events = set(event_total.keys()) | set(event_missing.keys())

    for event in all_events:
        total = event_total.get(event, 0)
        missing = event_missing.get(event, 0)
        missing_rate = round(missing / total * 100, 2) if total > 0 else 0

        result.append({
            "file": file,
            "event_type": event,
            "전체 행 수": total,
            "category_code 결측 수": missing,
            "category_code 결측 비율(%)": missing_rate
        })

In [19]:
category_missing_pattern_df = pd.DataFrame(result)
category_missing_pattern_df = category_missing_pattern_df.sort_values(["file", "event_type"]).reset_index(drop=True)

display(category_missing_pattern_df)

,file,event_type,전체 행 수,category_code 결측 수,category_code 결측 비율(%)
0,2019-Nov.csv,cart,3028930,826493,27.29
1,2019-Nov.csv,purchase,916939,234218,25.54
2,2019-Nov.csv,view,63556110,20837460,32.79
3,2019-Oct.csv,cart,926516,105726,11.41
4,2019-Oct.csv,purchase,742849,173425,23.35
5,2019-Oct.csv,view,40779399,13236458,32.46


- 10월 11월 둘 다 event_type에 차이는 있지만 높음
  - view에서 결측 비율이 가장 높음
  - 하지만 다른 cart와 purchase도 적지 않음
- product_id도 확인을 해보는 작업을 해야할듯
  - product_id는 같은 category_code를 가지는가
  - ex. 10000번째 번호면 어떤 카테고리이고 이런거
  - 관계를 확인해보면서 연관이 있으면 category_code 결측치를 채울수 있음

In [20]:
category_counter = Counter()

for file in files:
    for chunk in pd.read_csv(file, chunksize=chunksize, usecols=["category_code"]):
        counts = chunk["category_code"].value_counts(dropna=True).to_dict()
        category_counter.update(counts)

category_count_df = (
    pd.DataFrame(category_counter.items(), columns=["category_code", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(category_count_df.head(30))
print("전체 고유 category_code 개수:", len(category_count_df))

,category_code,count
0,electronics.smartphone,27882231
1,electronics.clocks,3397999
2,electronics.video.tv,3321796
3,computers.notebook,3318177
4,electronics.audio.headphone,2917065
5,apparel.shoes,2650791
6,appliances.environment.vacuum,2329728
7,appliances.kitchen.refrigerators,2314917
8,appliances.kitchen.washer,2273270
9,computers.desktop,1114744


전체 고유 category_code 개수: 129


In [21]:
result = []

for file in files:
    for chunk in pd.read_csv(file, chunksize=chunksize, usecols=["product_id", "category_code"]):
        temp = chunk.dropna(subset=["product_id", "category_code"]).copy()

        grouped = temp.groupby("category_code")["product_id"].agg(
            count="count",
            min="min",
            q1=lambda x: x.quantile(0.25),
            median="median",
            q3=lambda x: x.quantile(0.75),
            max="max"
        ).reset_index()

        result.append(grouped)

product_range_df = pd.concat(result, ignore_index=True)

# 청크별 결과를 다시 category_code 기준으로 합치기
product_range_summary = (
    product_range_df.groupby("category_code")
    .agg({
        "count": "sum",
        "min": "min",
        "q1": "median",
        "median": "median",
        "q3": "median",
        "max": "max"
    })
    .reset_index()
    .sort_values("count", ascending=False)
)

display(product_range_summary.head(30))

,category_code,count,min,q1,median,q3,max
100,electronics.smartphone,27882231,1000365,1004564.00,1004856.0,1005115.0,100023500
99,electronics.clocks,3397999,5100059,5100576.00,5100865.0,21404650.0,100027908
104,electronics.video.tv,3321796,1800002,1801690.00,1801832.0,1801929.0,100028487
72,computers.notebook,3318177,1300742,1306686.00,1307135.0,1307344.0,100027954
93,electronics.audio.headphone,2917065,4800013,4803428.00,4804056.0,4804295.0,100023687
12,apparel.shoes,2650791,20400016,28716532.00,28718755.0,28720700.0,100028537
29,appliances.environment.vacuum,2329728,3700006,3700755.00,3700926.0,3701178.0,100028395
47,appliances.kitchen.refrigerators,2314917,2700018,2701639.00,2702255.0,2800297.0,100027868
50,appliances.kitchen.washer,2273270,3600025,3600666.00,3601250.0,3601472.0,100027109
70,computers.desktop,1114744,1400368,1480394.00,1480707.0,17000029.5,100028221


Q1, Q3를 보면 카테고리별 연관성이 있어보이는데, 
max가 눈에 띄게 높음 => 이상치 

In [22]:
# 상위 카테고리 기준으로 product_id 이상치 확인
# q1, q3, IQR, upper_bound를 이용해서 max가 일부 이상치인지 판단

top_categories = category_count_df.head(10)["category_code"].tolist()
print("상위 10개 카테고리:")
display(category_count_df.head(10))

# 2) 상위 카테고리의 product_id 수집
top_data = []

for file in files:
    for chunk in pd.read_csv(file, chunksize=chunksize, usecols=["product_id", "category_code"]):
        temp = chunk[chunk["category_code"].isin(top_categories)].dropna(subset=["product_id", "category_code"]).copy()
        top_data.append(temp)

top_df = pd.concat(top_data, ignore_index=True)

# 3) 카테고리별 이상치 확인
result = []

for cat, group in top_df.groupby("category_code"):
    x = group["product_id"]

    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr
    lower_bound = q1 - 1.5 * iqr

    outlier_cnt = ((x < lower_bound) | (x > upper_bound)).sum()
    total_cnt = len(x)

    result.append({
        "category_code": cat,
        "count": total_cnt,
        "min": x.min(),
        "q1": q1,
        "median": x.median(),
        "q3": q3,
        "max": x.max(),
        "IQR": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_cnt,
        "outlier_rate(%)": round(outlier_cnt / total_cnt * 100, 4)
    })

outlier_check_df = pd.DataFrame(result).sort_values("count", ascending=False)
display(outlier_check_df)

상위 10개 카테고리:


,category_code,count
0,electronics.smartphone,27882231
1,electronics.clocks,3397999
2,electronics.video.tv,3321796
3,computers.notebook,3318177
4,electronics.audio.headphone,2917065
5,apparel.shoes,2650791
6,appliances.environment.vacuum,2329728
7,appliances.kitchen.refrigerators,2314917
8,appliances.kitchen.washer,2273270
9,computers.desktop,1114744


,category_code,count,min,q1,median,q3,max,IQR,lower_bound,upper_bound,outlier_count,outlier_rate(%)
8,electronics.smartphone,27882231,1000365,1004564.0,1004856.0,1005108.0,100023500,544.0,1003748.0,1005924.0,3356352,12.0376
7,electronics.clocks,3397999,5100059,5100576.0,5100864.0,21404838.0,100027908,16304262.0,-19355817.0,45861231.0,46630,1.3723
9,electronics.video.tv,3321796,1800002,1801690.0,1801848.0,1801940.0,100028487,250.0,1801315.0,1802315.0,188238,5.6668
5,computers.notebook,3318177,1300742,1306798.0,1307136.0,1307350.0,100027954,552.0,1305970.0,1308178.0,317041,9.5547
6,electronics.audio.headphone,2917065,4800013,4803459.0,4804056.0,4804295.0,100023687,836.0,4802205.0,4805549.0,352314,12.0777
0,apparel.shoes,2650791,20400016,28716520.0,28718723.0,28720700.0,100028537,4180.0,28710250.0,28726970.0,302002,11.3929
1,appliances.environment.vacuum,2329728,3700006,3700755.0,3700926.0,3701164.0,100028395,409.0,3700141.5,3701777.5,133594,5.7343
2,appliances.kitchen.refrigerators,2314917,2700018,2701639.0,2702256.0,2800350.0,100027868,98711.0,2553572.5,2948416.5,50849,2.1966
3,appliances.kitchen.washer,2273270,3600025,3600666.0,3601250.0,3601461.0,100027109,795.0,3599473.5,3602653.5,3234,0.1423
4,computers.desktop,1114744,1400368,1480396.0,1480708.0,17000026.0,100028221,15519630.0,-21799049.0,40279471.0,32518,2.9171


In [23]:
result = []

for file in files:
    event_total = {}
    event_missing = {}

    for chunk in pd.read_csv(file, chunksize=chunksize, usecols=["event_type", "brand"]):
        # event_type별 전체 건수
        total_counts = chunk["event_type"].value_counts(dropna=False).to_dict()
        for k, v in total_counts.items():
            event_total[k] = event_total.get(k, 0) + v

        # brand 결측인 행만 따로
        missing_chunk = chunk[chunk["brand"].isna()]
        missing_counts = missing_chunk["event_type"].value_counts(dropna=False).to_dict()
        for k, v in missing_counts.items():
            event_missing[k] = event_missing.get(k, 0) + v

    all_events = set(event_total.keys()) | set(event_missing.keys())

    for event in all_events:
        total = event_total.get(event, 0)
        missing = event_missing.get(event, 0)
        missing_rate = round(missing / total * 100, 2) if total > 0 else 0

        result.append({
            "file": file,
            "event_type": event,
            "전체 행 수": total,
            "brand 결측 수": missing,
            "brand 결측 비율(%)": missing_rate
        })

brand_missing_pattern_df = pd.DataFrame(result)
brand_missing_pattern_df = brand_missing_pattern_df.sort_values(["file", "event_type"]).reset_index(drop=True)

display(brand_missing_pattern_df)

,file,event_type,전체 행 수,brand 결측 수,brand 결측 비율(%)
0,2019-Nov.csv,cart,3028930,258531,8.54
1,2019-Nov.csv,purchase,916939,73362,8.00
2,2019-Nov.csv,view,63556110,8892185,13.99
3,2019-Oct.csv,cart,926516,18806,2.03
4,2019-Oct.csv,purchase,742849,58305,7.85
5,2019-Oct.csv,view,40779399,6039969,14.81


brand 결측은 view 비율이 높음
- 전체적으로 수치가 적어 unknown 처리 괜찮음

In [24]:
result = []

for file in files:
    group_total = {"smartphone": 0, "non_smartphone": 0}
    group_missing = {"smartphone": 0, "non_smartphone": 0}

    for chunk in pd.read_csv(file, chunksize=chunksize, usecols=["category_code", "brand"]):
        chunk["is_smartphone"] = chunk["category_code"].astype("string").str.contains(
            "smartphone", case=False, na=False
        )

        # 전체 건수
        total_counts = chunk["is_smartphone"].value_counts(dropna=False).to_dict()
        group_total["smartphone"] += total_counts.get(True, 0)
        group_total["non_smartphone"] += total_counts.get(False, 0)

        # brand 결측 건수
        missing_chunk = chunk[chunk["brand"].isna()]
        missing_counts = missing_chunk["is_smartphone"].value_counts(dropna=False).to_dict()
        group_missing["smartphone"] += missing_counts.get(True, 0)
        group_missing["non_smartphone"] += missing_counts.get(False, 0)

    for group_name, key in [("smartphone", "smartphone"), ("non_smartphone", "non_smartphone")]:
        total = group_total[key]
        missing = group_missing[key]
        missing_rate = round(missing / total * 100, 2) if total > 0 else 0

        result.append({
            "file": file,
            "group": group_name,
            "전체 행 수": total,
            "brand 결측 수": missing,
            "brand 결측 비율(%)": missing_rate
        })

brand_missing_smartphone_df = pd.DataFrame(result)
display(brand_missing_smartphone_df)

,file,group,전체 행 수,brand 결측 수,brand 결측 비율(%)
0,2019-Oct.csv,smartphone,11507231,21910,0.19
1,2019-Oct.csv,non_smartphone,30941533,6095170,19.70
2,2019-Nov.csv,smartphone,16375000,21418,0.13
3,2019-Nov.csv,non_smartphone,51126979,9202660,18.00


스마트폰 / 그외로 구분하여 브랜트 결측
- 스마트폰 브랜트 결측이 거의 없음
  - 0.13 ~ 0.19
- 그외 브랜드 결측이 나타남
  - 하지만 수치가 크지 않음

=> 브랜트 결측값 모두 'unknown'처리
  - 스마트폰 세부 분석 부분에서도 브랜드 없는 'unknown'로 분류

In [32]:
result = []

for file in files:
    total_rows = 0
    nonpositive_count = 0
    positive_count = 0

    min_price = None
    max_price = None
    all_price_stats = []

    for chunk in pd.read_csv(file, chunksize=chunksize, usecols=["price"]):
        price = pd.to_numeric(chunk["price"], errors="coerce")
        total_rows += len(price)

        nonpositive_count += (price <= 0).sum()
        pos = price[price > 0].dropna()

        if len(pos) == 0:
            continue

        positive_count += len(pos)

    stats_df = pd.DataFrame(all_price_stats)

    result.append({
        "file": file,
        "전체 행 수": total_rows,
        "price > 0 행 수": positive_count,
        "price <= 0 행 수": nonpositive_count,
        "price <= 0 비율(%)": round(nonpositive_count / total_rows * 100, 4)
    })

price_summary_df = pd.DataFrame(result)
display(price_summary_df)

,file,전체 행 수,price > 0 행 수,price <= 0 행 수,price <= 0 비율(%)
0,2019-Oct.csv,42448764,42380091,68673,0.1618
1,2019-Nov.csv,67501979,67313891,188088,0.2786


price <= 0 비율 낮음
  - 0.16 ~ 0.27%
  - 아주 일부에서 발생
월별 가격 분포 차이 없음

#### price 값이 -도 있는지
- 조건이 일치하지않다면 0값만 있다는 것

In [33]:
# 10월과 11월의 0 이하 데이터를 따로 저장할 리스트
result = []

for file in files:
    print(f"{file} 분석 중")
    reader = pd.read_csv(file, chunksize=chunksize)
    
    for chunk in reader:
        # price가 0 미만인 행만 골라내기
        cond = pd.to_numeric(chunk["price"], errors="coerce") < 0
        target_chunk = chunk[cond]
        
        if not target_chunk.empty:
            result.append(target_chunk)

# 하나로 합침
if result:
    df_negative = pd.concat(result, ignore_index=True)
    print("결과")
    print(f"전체 0 미만 행 수: {len(df_negative)}")
    display(df_negative.head())
else:
    print("조건에 맞는 데이터 없음")

2019-Oct.csv 분석 중
2019-Nov.csv 분석 중
조건에 맞는 데이터 없음


- price값 가장 작은 값은 0원이라는 결론
  - 이벤트성 상품
  - 환불 데이터
  - 데이터 수집 오류